
### Advanced Modelling — AfriBERTa Transformer Model

In this section, we implement an advanced multilingual transformer model using AfriBERTa to classify customer sentiment within Kenya’s mobile money ecosystem.

Unlike traditional machine learning models such as Logistic Regression and XGBoost, transformer models automatically learn contextual language representations and are better suited for multilingual and code-switched text containing English, Swahili, and Sheng.

The modelling workflow includes:

- Light text preprocessing
- AfriBERTa tokenization
- Transformer dataset preparation
- Fine-tuning the pretrained AfriBERTa model
- Model evaluation using classification metrics

The target variable is sentiment classification:
- Negative → 0
- Neutral → 1
- Positive → 2

Performance is evaluated using:
- Weighted F1-score
- Classification report
- Confusion matrix

The target performance is:
- Expected Weighted F1-score ≥ 0.82

This advanced model supports the project objective of building a real-time financial complaint monitoring system for Kenya’s fintech ecosystem.

In [1]:
#import libraries
import pandas as pd
import numpy as np

# Text cleaning
import re

# PyTorch

import torch

# Train-test split

from sklearn.model_selection import train_test_split
# Metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset

In [4]:
#load raw  dataset
df=pd.read_csv("MASTER_RAW_kenya_fintech.csv")
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,app_name
0,695de54e-4b85-4669-ae8c-ad2fdf16667e,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,"The app still has issues on OTP, because I hav...",1,0,5.1.7,2026-05-11 11:38:40,NaN,NaN,5.1.7,mpesa
1,acd5c061-de13-474b-8645-f628044f2a50,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,si everytime nitakuwa na bundles za ku check m...,2,0,5.1.1,2026-05-11 11:22:24,NaN,NaN,5.1.1,mpesa
2,6f9f52e9-0a00-4f70-a1cc-7687f28465a3,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,this is the stupidest app ever from saf. the w...,1,0,5.1.7,2026-05-11 11:16:47,NaN,NaN,5.1.7,mpesa
3,4a605b22-efc1-4641-b79e-e166b4a7b2e4,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,Life must go on without this useless app. It u...,1,0,1.14.2,2026-05-11 11:01:23,NaN,NaN,1.14.2,mpesa
4,fd284f23-d966-4be5-b421-a1f0e14c1e13,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,the upgrade is terrible,1,0,NaN,2026-05-11 10:45:52,NaN,NaN,NaN,mpesa


Functions are created to perform the following:

The dataset is lightly cleaned by:
- converting text to lowercase
- removing URLs
- removing extra spaces

Minimal preprocessing is applied for transformer models to perform better. 

Sentiment labels are created from star ratings

The dataset is then split into training and testing sets using stratified sampling to preserve class distribution.

Preperation for transformer training is done by transforming the text and labels tu Hugging Face Dataset format.

Evaluation is done


In [ ]:
def clean_text(text):
    
    text = str(text).lower()
    
    # Remove URLs
    
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove extra spaces
    
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply cleaning

df['clean_text'] = df['content'].apply(clean_text)


# Create sentiment labels


def create_sentiment(score):
    
    if score <= 2:
        return 0      # Negative
    
    elif score == 3:
        return 1      # Neutral
    
    else:
        return 2      # Positive

df['label'] = df['score'].apply(create_sentiment)

# Trainest-test split function

def split_data(data, text_column, target_column):
    
    X_train, X_test, y_train, y_test = train_test_split(
        data[text_column],
        data[target_column],
        test_size=0.2,
        stratify=data[target_column],
        random_state=42
    )
    
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = split_data(
    df,
    'clean_text',
    'label'
)

# CONVERT TO HUGGING FACE DATASET

def create_hf_dataset(texts, labels):
    
    dataset = Dataset.from_dict({
        'text': texts.tolist(),
        'label': labels.tolist()
    })
    
    return dataset

train_dataset = create_hf_dataset(X_train, y_train)

test_dataset = create_hf_dataset(X_test, y_test)

# Load tokenizer

model_name = "castorini/afriberta_large"

tokenizer = AutoTokenizer.from_pretrained(model_name)


# Create tokenization function

def tokenize_dataset(dataset, tokenizer):
    
    tokenized_dataset = dataset.map(
        lambda x: tokenizer(
            x['text'],
            padding='max_length',
            truncation=True,
            max_length=128
        ),
        batched=True
    )
    
    return tokenized_dataset

train_dataset = tokenize_dataset(
    train_dataset,
    tokenizer
)

test_dataset = tokenize_dataset(
    test_dataset,
    tokenizer
)

# load afriberta

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)


# Train arguments function

def create_training_arguments():
    
    training_args = TrainingArguments(
        
        output_dir='./results',
        
        evaluation_strategy='epoch',
        
        save_strategy='epoch',
        
        learning_rate=2e-5,
        
        per_device_train_batch_size=8,
        
        per_device_eval_batch_size=8,
        
        num_train_epochs=3,
        
        weight_decay=0.01,
        
        logging_dir='./logs',
        
        logging_steps=50
    )
    
    return training_args

training_args = create_training_arguments()


# Create Trainer function

def create_trainer(
    model,
    training_args,
    train_dataset,
    test_dataset
):
    
    trainer = Trainer(
        
        model=model,
        
        args=training_args,
        
        train_dataset=train_dataset,
        
        eval_dataset=test_dataset
    )
    
    return trainer

trainer = create_trainer(
    model,
    training_args,
    train_dataset,
    test_dataset
)

# Train the model
trainer.train()

# Create evaluation function

def evaluate_model(
    trainer,
    test_dataset,
    y_true,
    expected_f1=0.82
):
    
    predictions = trainer.predict(test_dataset)
    
    y_pred = predictions.predictions.argmax(axis=1)
    
    # Weighted F1
    
    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average='weighted'
    )
    
    print("Classification Report")
    
    print(classification_report(y_true, y_pred))
    
    print("Confusion Matrix")
    
    print(confusion_matrix(y_true, y_pred))
    
    print("Weighted F1 Score:")
    
    print(round(weighted_f1, 4))
    
    # Compare against expected target
    
    if weighted_f1 >= expected_f1:
        
        print(f"Model exceeded expected F1 target of {expected_f1}")
    
    else:
        
        print(f"Model did not reach expected F1 target of {expected_f1}")


# Run evaluation
evaluate_model(
    trainer,
    test_dataset,
    y_test
)